# Results Analysis

Analyses `metrics.csv` produced by `02_experiments.ipynb`.

**Structure:**
1. Load & overview
2. R² comparison across datasets and models
3. MAE and RMSE comparison
4. Feature group ablation (what actually helped?)
5. Best model per dataset
6. Conclusions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

METRICS_PATH = "E:\\code\\tour-prediction\\data\\results\\metrics.csv"

# Dataset order and labels
DATASET_ORDER = [
    "v1_baseline",
    "v2_artist",
    "v3_artist_geo",
    "v4_artist_geo_time",
    "v5_full",
    "v6_no_artist",
]

MODEL_COLORS = {
    "Decision Tree": "#4C8BBF",
    "Random Forest": "#2E9E6B",
    "MLP":           "#E07B3F",
}

## 1. Load & Overview

In [ ]:
metrics = pd.read_csv(METRICS_PATH)
metrics["dataset"] = pd.Categorical(metrics["dataset"], categories=DATASET_ORDER, ordered=True)
metrics = metrics.sort_values(["dataset", "model"]).reset_index(drop=True)

display(
    metrics
    .style
    .format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.4f}"})
    .background_gradient(subset=["R2"], cmap="RdYlGn")
    .background_gradient(subset=["MAE", "RMSE"], cmap="RdYlGn_r")
)

## 2. R² Comparison

Two views: grouped bar (easy to compare models within a dataset) and line chart (easy to track each model across datasets).

In [ ]:
pivot_r2 = metrics.pivot(index="dataset", columns="model", values="R2").reindex(DATASET_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Grouped bar ───────────────────────────────────────────────────────────────
ax = axes[0]
x = np.arange(len(DATASET_ORDER))
n_models = len(pivot_r2.columns)
width = 0.25
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

for offset, model in zip(offsets, pivot_r2.columns):
    vals = pivot_r2[model].values
    bars = ax.bar(x + offset, vals, width=width, label=model,
                  color=MODEL_COLORS.get(model, "gray"), alpha=0.88, edgecolor="white")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7, color="dimgray")

ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("R²", fontsize=11)
ax.set_ylim(0, 1.08)
ax.set_title("R² by Dataset and Model", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.25)
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)

# ── Line chart ────────────────────────────────────────────────────────────────
ax2 = axes[1]
for model in pivot_r2.columns:
    ax2.plot(DATASET_ORDER, pivot_r2[model].values, marker="o", linewidth=2,
             markersize=7, label=model, color=MODEL_COLORS.get(model, "gray"))

ax2.set_xticks(range(len(DATASET_ORDER)))
ax2.set_xticklabels(DATASET_ORDER, rotation=35, ha="right", fontsize=9)
ax2.set_ylabel("R²", fontsize=11)
ax2.set_title("R² Trend Across Dataset Versions", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 3. MAE and RMSE Comparison

R² can look good even when absolute errors are large. MAE and RMSE put errors back in attendance units (number of people), which is more interpretable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    pivot = metrics.pivot(index="dataset", columns="model", values=metric).reindex(DATASET_ORDER)

    x = np.arange(len(DATASET_ORDER))
    offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

    for offset, model in zip(offsets, pivot.columns):
        vals = pivot[model].values
        bars = ax.bar(x + offset, vals, width=width, label=model,
                      color=MODEL_COLORS.get(model, "gray"), alpha=0.88, edgecolor="white")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
                    f"{val:,.0f}", ha="center", va="bottom", fontsize=6.5, color="dimgray", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(DATASET_ORDER, rotation=35, ha="right", fontsize=9)
    ax.set_ylabel(f"{metric} (attendees)", fontsize=11)
    ax.set_title(f"{metric} by Dataset and Model", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## 4. Feature Group Ablation

Each dataset version adds a specific feature group. This section isolates the effect of each addition by comparing adjacent versions.

| Comparison | What changed |
|---|---|
| v1 → v2 | Added artist features |
| v2 → v3 | Added geo features |
| v3 → v4 | Added time features |
| v4 → v5 | Added remaining features (v5 = full) |
| v5 → v6 | Removed artist name |

In [ ]:
TRANSITIONS = [
    ("v1_baseline",       "v2_artist",          "+ artist features"),
    ("v2_artist",         "v3_artist_geo",       "+ geo features"),
    ("v3_artist_geo",     "v4_artist_geo_time",  "+ time features"),
    ("v4_artist_geo_time","v5_full",             "+ remaining features"),
    ("v5_full",           "v6_no_artist",        "− artist name"),
]

fig, axes = plt.subplots(1, len(pivot_r2.columns), figsize=(16, 5), sharey=True)

for ax, model in zip(axes, pivot_r2.columns):
    model_data = metrics[metrics["model"] == model].set_index("dataset")["R2"]

    labels, deltas, colors = [], [], []
    for v_from, v_to, label in TRANSITIONS:
        if v_from in model_data.index and v_to in model_data.index:
            delta = model_data[v_to] - model_data[v_from]
            labels.append(label)
            deltas.append(delta)
            colors.append("#2E9E6B" if delta >= 0 else "#D95F5F")

    bars = ax.barh(labels, deltas, color=colors, edgecolor="white", alpha=0.88)
    ax.axvline(0, color="black", linewidth=0.9)
    for bar, val in zip(bars, deltas):
        ax.text(val + (0.0003 if val >= 0 else -0.0003),
                bar.get_y() + bar.get_height() / 2,
                f"{val:+.4f}", va="center",
                ha="left" if val >= 0 else "right",
                fontsize=8, color="dimgray")
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_xlabel("ΔR²", fontsize=10)
    ax.grid(axis="x", alpha=0.2)

axes[0].set_ylabel("Feature group change", fontsize=10)
fig.suptitle("ΔR² per Feature Group Addition / Removal", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Best Model per Dataset

In [ ]:
best = (
    metrics
    .loc[metrics.groupby("dataset")["R2"].idxmax(), ["dataset", "model", "R2", "MAE", "RMSE"]]
    .reset_index(drop=True)
)

display(
    best
    .style
    .format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.4f}"})
    .background_gradient(subset=["R2"], cmap="RdYlGn")
    .background_gradient(subset=["MAE", "RMSE"], cmap="RdYlGn_r")
)

# Overall winner
winner = best.loc[best["R2"].idxmax()]
print(f"\nOverall best: {winner['model']} on {winner['dataset']}")
print(f"  R²={winner['R2']:.4f}  MAE={winner['MAE']:,.0f}  RMSE={winner['RMSE']:,.0f}")

---
## 6. Conclusions

Answer each question after running the cells above.

In [ ]:
# Auto-fill answers from the data

best_dataset = best.loc[best["R2"].idxmax(), "dataset"]
best_model_overall = metrics.groupby("model")["R2"].mean().idxmax()

def r2_for(dataset, model):
    row = metrics[(metrics["dataset"] == dataset) & (metrics["model"] == model)]
    return row["R2"].values[0] if len(row) else None

def delta(d1, d2, model="Random Forest"):
    a, b = r2_for(d1, model), r2_for(d2, model)
    return b - a if a is not None and b is not None else None

conclusions = {
    "1. Best dataset overall":              best_dataset,
    "2. Best model overall (avg R²)":       best_model_overall,
    "3. Artist features helped? (v1→v2)":   f"ΔR²={delta('v1_baseline','v2_artist'):+.4f}",
    "4. Geo features helped? (v2→v3)":      f"ΔR²={delta('v2_artist','v3_artist_geo'):+.4f}",
    "5. Time features helped? (v3→v4)":     f"ΔR²={delta('v3_artist_geo','v4_artist_geo_time'):+.4f}",
    "6. Removing artist name hurt? (v5→v6)": f"ΔR²={delta('v5_full','v6_no_artist'):+.4f}",
}

print("CONCLUSIONS")
print("=" * 55)
for question, answer in conclusions.items():
    print(f"  {question}")
    print(f"    → {answer}")
    print()